In [ ]:
import cv2
import os

# --- CONFIGURATION ---
SAVE_PATH = "dataset"
IMAGES_PER_POSE = 5  # Number of shots per angle
TOTAL_REQUIRED = IMAGES_PER_POSE * 5 

# Load the Face Detection Model
# Note: Ensure opencv-python is installed (pip install opencv-python)
face_classifier = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def sanitize_filename(text):
    """
    Replaces forward slashes with hyphens to prevent folder errors.
    Input: "NBTI/PER/1190" -> Output: "NBTI-PER-1190"
    """
    return text.replace("/", "-").replace("\\", "-")

def get_pose_instruction(count, limit):
    """
    Returns instructions based on how many photos we have taken.
    """
    if count < limit:
        return "LOOK STRAIGHT"
    elif count < limit * 2:
        return "TURN SLIGHTLY LEFT"
    elif count < limit * 3:
        return "TURN SLIGHTLY RIGHT"
    elif count < limit * 4:
        return "CHIN UP (Look Up)"
    else:
        return "CHIN DOWN (Look Down)"

def collect_data_manual():
    print("--- MANUAL STAFF ENROLLMENT (MIRRORED) ---")
    
    # 1. Get User Info
    name = input("Enter Staff Name: ").strip()
    staff_id = input("Enter Staff ID: ").strip()
    
    # 2. Prepare Directory
    safe_id = sanitize_filename(staff_id)
    folder_name = f"{name}_{safe_id}"
    path = os.path.join(SAVE_PATH, folder_name)
    
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"[INFO] New folder created: {path}")
    else:
        print(f"[INFO] Adding to existing folder: {path}")

    # 3. Start Camera
    cam = cv2.VideoCapture(0)
    count = 0
    
    window_name = "Mirrored Enrollment"
    
    print(f"\n[START] System ready. We need {TOTAL_REQUIRED} photos.")
    print("[CONTROLS] SPACE: Capture | Q: Quit")
    
    while True:
        ret, frame = cam.read()
        if not ret:
            print("[ERROR] Camera not found or disconnected.")
            break

        # --- MIRROR THE FRAME ---
        frame = cv2.flip(frame, 1)

        # --- CHECK IF WINDOW CLOSED ('X' BUTTON) ---
        try:
            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                print("Window closed manually.")
                break
        except Exception:
            pass

        # Detect Faces
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3, 5)

        # UI Logic
        instruction = get_pose_instruction(count, IMAGES_PER_POSE)
        
        # Draw Progress (Top Left)
        cv2.putText(frame, f"Progress: {count}/{TOTAL_REQUIRED}", (30, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        
        # Draw Instruction (Bottom Left)
        color = (0, 0, 255) if count < TOTAL_REQUIRED else (0, 255, 0)
        msg = instruction if count < TOTAL_REQUIRED else "ENROLLMENT COMPLETE!"
        cv2.putText(frame, msg, (30, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

        face_detected = False
        target_face = None
        
        # Draw Box around Face
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            face_detected = True
            # Crop the face for saving
            target_face = gray[y:y+h, x:x+w]

        cv2.imshow(window_name, frame)

        # --- KEYBOARD CONTROLS ---
        k = cv2.waitKey(1) & 0xFF

        # Quit on 'q' or ESC
        if k == ord('q') or k == 27: 
            print("Quitting...")
            break
        
        # Capture on SPACE
        elif k == 32: 
            if not face_detected:
                print("[WARNING] No face detected! Look at the camera.")
                continue
                
            if count >= TOTAL_REQUIRED:
                print("Data collection finished. Press 'Q' to exit.")
                continue

            count += 1
            file_name_path = f"{path}/User.{safe_id}.{count}.jpg"
            cv2.imwrite(file_name_path, target_face)
            print(f"[SAVED] {count}/{TOTAL_REQUIRED} | Pose: {instruction}")
            
            # Flash Visual Effect
            cv2.rectangle(frame, (0,0), (640,480), (255,255,255), cv2.FILLED)
            cv2.imshow(window_name, frame)
            cv2.waitKey(50) # Small pause for flash

    cam.release()
    cv2.destroyAllWindows()
    print("--- SESSION CLOSED ---")

if __name__ == "__main__":
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
    collect_data_manual()

--- MANUAL STAFF ENROLLMENT (MIRRORED) ---

[START] System ready. We need 25 photos.
[INSTRUCTION] Align your face, then PRESS SPACEBAR to capture.
[SAVED] 1/25 | Pose: LOOK STRAIGHT
[SAVED] 2/25 | Pose: LOOK STRAIGHT
[SAVED] 3/25 | Pose: LOOK STRAIGHT
[SAVED] 4/25 | Pose: LOOK STRAIGHT
[SAVED] 5/25 | Pose: LOOK STRAIGHT
[SAVED] 6/25 | Pose: TURN SLIGHTLY LEFT
[SAVED] 7/25 | Pose: TURN SLIGHTLY LEFT
[WARNING] No face detected!
[SAVED] 8/25 | Pose: TURN SLIGHTLY LEFT
[SAVED] 9/25 | Pose: TURN SLIGHTLY LEFT
[SAVED] 10/25 | Pose: TURN SLIGHTLY LEFT
[SAVED] 11/25 | Pose: TURN SLIGHTLY RIGHT
[WARNING] No face detected!
[WARNING] No face detected!
[WARNING] No face detected!
[WARNING] No face detected!
[SAVED] 12/25 | Pose: TURN SLIGHTLY RIGHT
[SAVED] 13/25 | Pose: TURN SLIGHTLY RIGHT
[SAVED] 14/25 | Pose: TURN SLIGHTLY RIGHT
[SAVED] 15/25 | Pose: TURN SLIGHTLY RIGHT
[SAVED] 16/25 | Pose: CHIN UP (Look Up)
[WARNING] No face detected!
[WARNING] No face detected!
[WARNING] No face detected!
[S

: 

In [1]:
import cv2
import cvzone
from cvzone.FaceDetectionModule import FaceDetector
import os

# --- CONFIGURATION ---
save_path = 'ImagesAttendance'
offset = 40  # Padding around the face (pixels) for a better crop
confidence_threshold = 0.8 # Only save high-quality detections

# Ensure folder exists
if not os.path.exists(save_path):
    os.makedirs(save_path)

# --- 1. GET USER DETAILS (CONSOLE) ---
print("--- NEW STAFF REGISTRATION ---")
name = input("Enter Full Name (e.g., ElonMusk): ").strip().replace(" ", "")
dept = input("Enter Department (e.g., IT): ").strip().replace(" ", "")
staff_id = input("Enter Staff ID (e.g., 001): ").strip()

filename = f"{name}_{dept}_{staff_id}.jpg"
full_path = f"{save_path}/{filename}"

# --- 2. SETUP CAMERA ---
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

detector = FaceDetector(minDetectionCon=0.75)

print("\n[INSTRUCTIONS]")
print("1. Look at the camera.")
print("2. Align your face inside the box.")
print("3. Press 's' to SAVE and Exit.")
print("4. Press 'q' to Quit without saving.")

while True:
    success, img = cap.read()
    img = cv2.flip(img, 1) # Mirror
    img_clean = img.copy() # Keep a clean version for saving

    # Detect Face
    img, bboxs = detector.findFaces(img, draw=False)

    face_valid = False
    
    if bboxs:
        # Get the largest face (closest to camera)
        x, y, w, h = bboxs[0]['bbox']
        score = bboxs[0]['score'][0]
        
        # Check if face is centered and clear
        if score > confidence_threshold:
            face_valid = True
            
            # Draw Green "Good to Save" Box
            cv2.rectangle(img, (x - offset, y - offset), (x + w + offset, y + h + offset), (0, 255, 0), 2)
            cv2.putText(img, "PERFECT! Press 's'", (x, y - 20), cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 2)
        else:
            # Face too blurry or far
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cv2.putText(img, "Get Closer / Hold Still", (x, y - 20), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2)

    else:
        cv2.putText(img, "No Face Detected", (50, 50), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2)

    # Show Preview
    cv2.imshow("Data Collector", img)
    
    key = cv2.waitKey(1)
    
    # --- SAVE LOGIC ---
    if key == ord('s'):
        if face_valid and bboxs:
            try:
                # Crop the face with padding (Passport Style)
                x, y, w, h = bboxs[0]['bbox']
                
                # Ensure we don't crop outside image boundaries
                y1 = max(0, y - offset)
                y2 = min(img_clean.shape[0], y + h + offset)
                x1 = max(0, x - offset)
                x2 = min(img_clean.shape[1], x + w + offset)
                
                imgCrop = img_clean[y1:y2, x1:x2]
                
                # Resize to standard size (optional, but good for consistency)
                imgCrop = cv2.resize(imgCrop, (300, 300))
                
                cv2.imwrite(full_path, imgCrop)
                print(f"\n[SUCCESS] Saved: {filename}")
                break
            except Exception as e:
                print(f"[ERROR] Could not crop/save: {e}")
        else:
            print("[WARNING] Face not clear enough to save! Try again.")

    if key == ord('q'):
        print("[INFO] Cancelled.")
        break

cap.release()
cv2.destroyAllWindows()

--- NEW STAFF REGISTRATION ---

[INSTRUCTIONS]
1. Look at the camera.
2. Align your face inside the box.
3. Press 's' to SAVE and Exit.
4. Press 'q' to Quit without saving.

[SUCCESS] Saved: BELLOMUHAMMAD_ICT_NBTI/PER/1190.jpg


In [2]:
import cv2
import os
from cvzone.FaceDetectionModule import FaceDetector

# --- CONFIGURATION ---
DATASET_PATH = "dataset"            # For storing all raw angles (Backup/Training)
SYSTEM_PATH = "ImagesAttendance"    # For the actual Attendance System (The one file it needs)
IMAGES_PER_POSE = 5                 # Number of shots per angle
TOTAL_REQUIRED = IMAGES_PER_POSE * 5 
OFFSET = 40                         # Padding around face (Passport Style)

# Initialize the "Smart" Face Detector
# minDetectionCon=0.75 means it only accepts high-quality faces
detector = FaceDetector(minDetectionCon=0.75)

def sanitize_filename(text):
    """Clean text for folder names."""
    return text.replace("/", "-").replace("\\", "-").strip()

def get_pose_instruction(count, limit):
    """Tell the user where to look based on progress."""
    if count < limit:
        return "LOOK STRAIGHT (Frontal)"
    elif count < limit * 2:
        return "TURN SLIGHTLY LEFT"
    elif count < limit * 3:
        return "TURN SLIGHTLY RIGHT"
    elif count < limit * 4:
        return "CHIN UP (Look Up)"
    else:
        return "CHIN DOWN (Look Down)"

def collect_data_smart():
    print("--- SMART STAFF ENROLLMENT ---")
    
    # 1. Get User Info
    name = input("Enter Staff Name (e.g., ElonMusk): ").strip().replace(" ", "")
    dept = input("Enter Department (e.g., IT): ").strip().replace(" ", "")
    staff_id = input("Enter Staff ID (e.g., 001): ").strip()
    
    safe_id = sanitize_filename(staff_id)
    
    # 2. Prepare Directories
    # A. Dataset Folder (For all 25 images)
    folder_name = f"{name}_{safe_id}"
    dataset_full_path = os.path.join(DATASET_PATH, folder_name)
    
    if not os.path.exists(dataset_full_path):
        os.makedirs(dataset_full_path)
    
    # B. System Folder (Ensure the main app folder exists)
    if not os.path.exists(SYSTEM_PATH):
        os.makedirs(SYSTEM_PATH)

    # 3. Start Camera
    cap = cv2.VideoCapture(0)
    cap.set(3, 1280) # High Res
    cap.set(4, 720)
    
    count = 0
    window_name = "Smart Enrollment"
    
    print(f"\n[START] System ready. We need {TOTAL_REQUIRED} photos.")
    print("[CONTROLS] SPACE: Capture | Q: Quit")
    
    while True:
        success, img = cap.read()
        if not success:
            print("[ERROR] Camera not found.")
            break

        # --- MIRROR THE FRAME ---
        img = cv2.flip(img, 1)
        img_clean = img.copy() # Keep a clean copy for saving

        # --- DETECT FACES (Smart Way) ---
        img, bboxs = detector.findFaces(img, draw=False)
        
        face_valid = False
        target_crop = None

        if bboxs:
            # Get the largest face
            x, y, w, h = bboxs[0]['bbox']
            score = bboxs[0]['score'][0]
            
            # --- QUALITY CHECK UI ---
            # Draw a box with corner logic
            color = (0, 255, 0) # Green by default
            
            # Logic: Ensure face is fully inside frame for a good crop
            if y - OFFSET < 0 or x - OFFSET < 0 or y+h+OFFSET > img.shape[0] or x+w+OFFSET > img.shape[1]:
                color = (0, 0, 255) # Red (Too close to edge)
                cv2.putText(img, "CENTER YOUR FACE", (x, y - 50), cv2.FONT_HERSHEY_DUPLEX, 0.8, color, 2)
            else:
                face_valid = True
                
                # Create the Passport Crop
                y1, y2 = y - OFFSET, y + h + OFFSET
                x1, x2 = x - OFFSET, x + w + OFFSET
                target_crop = img_clean[y1:y2, x1:x2]

            # Draw UI Box
            cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
            cv2.rectangle(img, (x, y - 40), (x + w, y), color, cv2.FILLED)
            cv2.putText(img, f"{int(score*100)}%", (x + 10, y - 10), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

        # --- PROGRESS & INSTRUCTIONS ---
        instruction = get_pose_instruction(count, IMAGES_PER_POSE)
        
        # Top Bar
        cv2.rectangle(img, (0, 0), (1280, 80), (20, 20, 20), cv2.FILLED)
        cv2.putText(img, f"Progress: {count}/{TOTAL_REQUIRED}", (30, 50), cv2.FONT_HERSHEY_DUPLEX, 1, (0, 255, 255), 2)
        
        # Bottom Bar (Instructions)
        msg_color = (0, 255, 0) if count >= TOTAL_REQUIRED else (0, 255, 255)
        msg_text = "ENROLLMENT COMPLETE! Press Q" if count >= TOTAL_REQUIRED else instruction
        
        cv2.putText(img, msg_text, (30, 680), cv2.FONT_HERSHEY_DUPLEX, 1.2, msg_color, 2)

        cv2.imshow(window_name, img)

        # --- CONTROLS ---
        k = cv2.waitKey(1) & 0xFF

        if k == ord('q'):
            break
        
        elif k == 32: # SPACE to Capture
            if count >= TOTAL_REQUIRED:
                print("Enrollment finished. Press Q to exit.")
                continue

            if face_valid and target_crop is not None:
                count += 1
                
                # 1. Save to Dataset (History/Backup)
                # Format: User.ID.Count.jpg
                filename_dataset = f"User.{safe_id}.{count}.jpg"
                path_dataset = os.path.join(dataset_full_path, filename_dataset)
                cv2.imwrite(path_dataset, target_crop)
                
                # 2. AUTO-SAVE FOR ATTENDANCE SYSTEM (Crucial Step)
                # We save the VERY FIRST "Look Straight" photo to the main folder
                # naming it exactly how the Attendance System needs it: Name_Dept_ID.jpg
                if count == 1:
                    filename_system = f"{name}_{dept}_{safe_id}.jpg"
                    path_system = os.path.join(SYSTEM_PATH, filename_system)
                    cv2.imwrite(path_system, target_crop)
                    print(f"[SYSTEM] Main Reference Photo Updated: {filename_system}")

                print(f"[SAVED] {count}/{TOTAL_REQUIRED} - {instruction}")
                
                # Visual Flash
                cv2.rectangle(img, (0,0), (1280,720), (255,255,255), cv2.FILLED)
                cv2.imshow(window_name, img)
                cv2.waitKey(100)
            
            else:
                print("[WARNING] Face not valid or too close to edge!")

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    collect_data_smart()

--- SMART STAFF ENROLLMENT ---

[START] System ready. We need 25 photos.
[CONTROLS] SPACE: Capture | Q: Quit
[SYSTEM] Main Reference Photo Updated: BELLOMUSTAPHA_ICT_NBTI-PER-1190.jpg
[SAVED] 1/25 - LOOK STRAIGHT (Frontal)
[SAVED] 2/25 - LOOK STRAIGHT (Frontal)
[SAVED] 3/25 - LOOK STRAIGHT (Frontal)
[SAVED] 4/25 - LOOK STRAIGHT (Frontal)
[SAVED] 5/25 - LOOK STRAIGHT (Frontal)
[SAVED] 6/25 - TURN SLIGHTLY LEFT
[SAVED] 7/25 - TURN SLIGHTLY LEFT
[SAVED] 8/25 - TURN SLIGHTLY LEFT
[SAVED] 9/25 - TURN SLIGHTLY LEFT
[SAVED] 10/25 - TURN SLIGHTLY LEFT
[SAVED] 11/25 - TURN SLIGHTLY RIGHT
[SAVED] 12/25 - TURN SLIGHTLY RIGHT
[SAVED] 13/25 - TURN SLIGHTLY RIGHT
[SAVED] 14/25 - TURN SLIGHTLY RIGHT
[SAVED] 15/25 - TURN SLIGHTLY RIGHT
[SAVED] 16/25 - CHIN UP (Look Up)
[SAVED] 17/25 - CHIN UP (Look Up)
[SAVED] 18/25 - CHIN UP (Look Up)
[SAVED] 19/25 - CHIN UP (Look Up)
[SAVED] 20/25 - CHIN UP (Look Up)
[SAVED] 21/25 - CHIN DOWN (Look Down)
[SAVED] 22/25 - CHIN DOWN (Look Down)
[SAVED] 23/25 - CHIN DOW

In [6]:
import cv2
import os
import time
from cvzone.FaceDetectionModule import FaceDetector

# --- CONFIGURATION ---
DATASET_DIR = 'dataset'             # For Training (Multiple photos)
REFERENCE_DIR = 'ImagesAttendance'  # For Attendance System (1 photo)
IMAGES_TO_TAKE = 10                 # How many dataset photos to take
OFFSET = 20                         # Padding around face (Passport style)

# Initialize Detector (High accuracy)
detector = FaceDetector(minDetectionCon=0.75)

# --- HELPER FUNCTIONS ---
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def clean_text(text):
    """Removes spaces and special chars."""
    return "".join([c for c in text if c.isalnum() or c in (' ', '_', '-')]).strip().replace(" ", "")

# --- MAIN SYSTEM ---
def main_system():
    # Setup Folders
    create_dir(DATASET_DIR)
    create_dir(REFERENCE_DIR)

    while True:
        # --- 1. MAIN MENU / INPUTS ---
        print("\n" + "="*40)
        print("   NEW STAFF REGISTRATION SYSTEM")
        print("   Type 'exit' to close the program.")
        print("="*40)

        name = input("1. Enter Full Name  : ").strip()
        if name.lower() == 'exit': break
        if not name: continue # skip if empty

        dept = input("2. Enter Department : ").strip()
        if dept.lower() == 'exit': break

        staff_id = input("3. Enter Staff ID   : ").strip()
        if staff_id.lower() == 'exit': break

        # Sanitize Inputs
        safe_name = clean_text(name)
        safe_dept = clean_text(dept)
        safe_id = clean_text(staff_id)

        # Folder Name: Name_Dept_ID
        folder_name = f"{safe_name}_{safe_dept}_{safe_id}"
        
        # Paths
        person_dataset_path = os.path.join(DATASET_DIR, folder_name)
        system_file_name = f"{folder_name}.jpg"
        system_file_path = os.path.join(REFERENCE_DIR, system_file_name)

        print(f"\n[INFO] Get ready to capture for: {safe_name}")
        print("[INSTRUCTIONS] Look at camera. Press 's' to START. Press 'q' to CANCEL/GO BACK.")
        input("Press Enter to open camera...")

        # --- 2. CAMERA LOOP ---
        cap = cv2.VideoCapture(0)
        cap.set(3, 1280) # Width
        cap.set(4, 720)  # Height
        
        count = 0
        capturing = False
        
        while True:
            success, img = cap.read()
            if not success: break
            
            img = cv2.flip(img, 1)
            img_clean = img.copy() # Copy for saving
            
            # Detect Face
            img, bboxs = detector.findFaces(img, draw=False)
            
            face_valid = False
            target_crop = None

            if bboxs:
                # Get Largest Face
                x, y, w, h = bboxs[0]['bbox']
                
                # Check boundaries (Don't save if face is cut off)
                if x - OFFSET > 0 and y - OFFSET > 0 and \
                   x + w + OFFSET < img.shape[1] and y + h + OFFSET < img.shape[0]:
                    
                    face_valid = True
                    
                    # Prepare Crop (Passport Style)
                    target_crop = img_clean[y-OFFSET : y+h+OFFSET, x-OFFSET : x+w+OFFSET]
                    
                    # Visual: Green Box
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                else:
                    # Visual: Red Box (Too close to edge)
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)
                    cv2.putText(img, "Move to Center", (x, y-20), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2)

            # --- UI TEXT ---
            if capturing:
                cv2.putText(img, f"Capturing: {count}/{IMAGES_TO_TAKE}", (50, 50), 
                            cv2.FONT_HERSHEY_DUPLEX, 1, (0, 165, 255), 2)
            else:
                cv2.putText(img, "Press 's' to Start | 'q' to Cancel", (50, 50), 
                            cv2.FONT_HERSHEY_DUPLEX, 0.8, (200, 200, 200), 2)

            cv2.imshow("Registration - 'q' to Go Back", img)
            
            # --- KEYBOARD CONTROLS ---
            key = cv2.waitKey(1) & 0xFF

            # 1. CANCEL (Go back to menu)
            if key == ord('q'):
                print("[INFO] Capture Cancelled. Going back to menu...")
                # Remove folder if we started creating it but cancelled
                if os.path.exists(person_dataset_path) and count == 0:
                    os.rmdir(person_dataset_path)
                break 

            # 2. START CAPTURING
            if key == ord('s'):
                capturing = True

            # 3. AUTOMATIC CAPTURE LOGIC
            if capturing and face_valid and target_crop is not None:
                # Create user folder only when we actually start saving
                if count == 0:
                    create_dir(person_dataset_path)

                count += 1
                
                # A. Save to Dataset (User_Name_Dept_ID_1.jpg)
                p_path = os.path.join(person_dataset_path, f"{folder_name}_{count}.jpg")
                cv2.imwrite(p_path, target_crop)
                
                # B. Save Reference for System (Only the 1st photo)
                if count == 1:
                    cv2.imwrite(system_file_path, target_crop)
                    print(f"\n[SUCCESS] System Reference Saved: {system_file_name}")

                print(f"Captured {count}/{IMAGES_TO_TAKE}")
                
                # Visual Feedback (White Flash)
                cv2.imshow("Registration - 'q' to Go Back", np.full_like(img, 255))
                cv2.waitKey(50) # Tiny pause

                if count >= IMAGES_TO_TAKE:
                    print(f"[DONE] {safe_name} registered successfully!")
                    cv2.waitKey(1000) # Show success for 1 sec
                    break
        
        # Cleanup Camera for this user
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    import numpy as np # Needed for flash effect
    main_system()


   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

[INFO] Get ready to capture for: AdamuDavid
[INSTRUCTIONS] Look at camera. Press 's' to START. Press 'q' to CANCEL/GO BACK.

[SUCCESS] System Reference Saved: AdamuDavid_ICT_NBTIPER1197.jpg
Captured 1/10
Captured 2/10
Captured 3/10
Captured 4/10
Captured 5/10
Captured 6/10
Captured 7/10
Captured 8/10
Captured 9/10
Captured 10/10
[DONE] AdamuDavid registered successfully!

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.

   NEW STAFF REGISTRATION SYSTEM
   Type 'exit' to close the program.


In [9]:
import cv2
import os
import time
import numpy as np
from cvzone.FaceDetectionModule import FaceDetector

# --- CONFIGURATION ---
# DIRECT PATH TO IMAGES ATTENDANCE (No more 'dataset' folder)
DESTINATION_DIR = r'C:\Users\hello\Desktop\NBTI PROJECTS\ICT Department Facial Recognition\ImagesAttendance'

IMAGES_TO_TAKE = 10                 # How many photos to take
OFFSET = 20                         # Padding around face

# Initialize Detector
detector = FaceDetector(minDetectionCon=0.75)

# --- HELPER FUNCTIONS ---
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def clean_text(text):
    """Removes spaces and special chars."""
    return "".join([c for c in text if c.isalnum() or c in (' ', '_', '-')]).strip().replace(" ", "")

# --- MAIN SYSTEM ---
def main_system():
    # Setup Main Folder
    create_dir(DESTINATION_DIR)

    while True:
        # --- 1. MAIN MENU ---
        print("\n" + "="*50)
        print("   NBTI STAFF REGISTRATION (Direct to Attendance)")
        print("   Type 'exit' to close.")
        print("="*50)

        name = input("1. Enter Full Name      : ").strip()
        if name.lower() == 'exit': break
        if not name: continue 

        dept = input("2. Enter Department     : ").strip()
        if dept.lower() == 'exit': break

        role = input("3. Enter Job Role       : ").strip()
        if role.lower() == 'exit': break

        staff_id = input("4. Enter Staff ID       : ").strip()
        if staff_id.lower() == 'exit': break

        # Sanitize Inputs
        safe_name = clean_text(name)
        safe_dept = clean_text(dept)
        safe_role = clean_text(role)
        safe_id = clean_text(staff_id)

        # Folder Name: Name_Department_Role_ID
        folder_name = f"{safe_name}_{safe_dept}_{safe_role}_{safe_id}"
        
        # FINAL PATH: ImagesAttendance / User_Folder
        user_folder_path = os.path.join(DESTINATION_DIR, folder_name)

        print(f"\n[INFO] Target Folder: {user_folder_path}")
        print("[INSTRUCTIONS] Press 's' to START. Press 'q' to CANCEL.")
        input("Press Enter to open camera...")

        # --- 2. CAMERA LOOP ---
        cap = cv2.VideoCapture(0)
        cap.set(3, 1280) 
        cap.set(4, 720) 
        
        count = 0
        capturing = False
        
        while True:
            success, img = cap.read()
            if not success: break
            
            img = cv2.flip(img, 1)
            img_clean = img.copy()
            
            # Detect Face
            img, bboxs = detector.findFaces(img, draw=False)
            
            face_valid = False
            target_crop = None

            if bboxs:
                x, y, w, h = bboxs[0]['bbox']
                
                # Check boundaries
                if x - OFFSET > 0 and y - OFFSET > 0 and \
                   x + w + OFFSET < img.shape[1] and y + h + OFFSET < img.shape[0]:
                    
                    face_valid = True
                    target_crop = img_clean[y-OFFSET : y+h+OFFSET, x-OFFSET : x+w+OFFSET]
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                else:
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)

            # --- UI TEXT ---
            if capturing:
                cv2.putText(img, f"Saving to Attendance: {count}/{IMAGES_TO_TAKE}", (50, 50), 
                            cv2.FONT_HERSHEY_DUPLEX, 1, (0, 165, 255), 2)
            else:
                cv2.putText(img, "Press 's' to Start | 'q' to Cancel", (50, 50), 
                            cv2.FONT_HERSHEY_DUPLEX, 0.8, (200, 200, 200), 2)

            cv2.imshow("Registration", img)
            
            key = cv2.waitKey(1) & 0xFF

            if key == ord('q'):
                print("[INFO] Cancelled.")
                if os.path.exists(user_folder_path) and count == 0:
                    os.rmdir(user_folder_path)
                break 

            if key == ord('s'):
                capturing = True

            if capturing and face_valid and target_crop is not None:
                # Create the user folder inside ImagesAttendance now
                if count == 0:
                    create_dir(user_folder_path)

                count += 1
                
                # SAVE DIRECTLY TO IMAGES ATTENDANCE FOLDER
                file_name = f"{folder_name}_{count}.jpg"
                save_path = os.path.join(user_folder_path, file_name)
                
                cv2.imwrite(save_path, target_crop)
                
                print(f"Saved: {file_name}")
                
                cv2.imshow("Registration", np.full_like(img, 255)) # Flash
                cv2.waitKey(50) 

                if count >= IMAGES_TO_TAKE:
                    print(f"[DONE] All images saved to ImagesAttendance/{folder_name}")
                    cv2.waitKey(1000) 
                    break
        
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main_system()


   NBTI STAFF REGISTRATION (Direct to Attendance)
   Type 'exit' to close.

[INFO] Target Folder: C:\Users\hello\Desktop\NBTI PROJECTS\ICT Department Facial Recognition\ImagesAttendance\BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190
[INSTRUCTIONS] Press 's' to START. Press 'q' to CANCEL.
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_1.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_2.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_3.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_4.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_5.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_6.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_7.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_8.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_9.jpg
Saved: BelloMuhammadMustapha_ICT_ProgrammeAnalystII_NBTIPER1190_10.jpg